# Fase 1 — Descarga Sentinel-2 Mandarina Murcott
**Objetivo**: Descargar imágenes Sentinel-2 L2A en color natural (RGB) para abril 2025 y abril 2026,
visualizarlas, identificar cobertura de nubes y exportar a PNG + GeoTIFF.

**ROI**: Shapefile desde Google Drive
**Bandas**: B2 (Blue), B3 (Green), B4 (Red) — Color natural
**Mes**: Abril 2025 y Abril 2026

In [ ]:
# =============================================================================
# CELDA 1: INSTALACIÓN DE DEPENDENCIAS
# =============================================================================
!pip install pystac-client stackstac rioxarray geopandas rasterio pystac -q
!pip install odc-stac -q
print("✅ Dependencias instaladas.")

In [ ]:
# =============================================================================
# CELDA 2: IMPORTACIONES
# =============================================================================
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from collections import Counter
from pystac_client import Client
import stackstac
from odc.stac import load
from google.colab import drive

print("✅ Importaciones completadas.")

In [ ]:
# =============================================================================
# CELDA 3: MONTAR GOOGLE DRIVE
# =============================================================================
drive.mount('/content/drive')
print("✅ Google Drive montado correctamente.")

In [ ]:
# =============================================================================
# CELDA 4: FUNCIONES AUXILIARES
# =============================================================================

def tif_to_png_batch(input_dir, output_dir, cmap_name='viridis', percentiles=(2, 98)):
    """
    Convierte archivos TIF a PNG con optimización de contraste.
    Útil para índices espectrales de una sola banda.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 Carpeta creada: {output_dir}")

    files = sorted(glob.glob(os.path.join(input_dir, "*.tif")))
    print(f"🚀 Procesando {len(files)} archivos...")

    for tif_path in files:
        filename = os.path.basename(tif_path).replace('.tif', '.png')
        save_path = os.path.join(output_dir, filename)

        with rasterio.open(tif_path) as src:
            data = src.read(1).astype(np.float32)
            nodata = src.nodata
            if nodata is not None:
                data[data == nodata] = np.nan

        vmin, vmax = np.nanpercentile(data, percentiles)
        norm = Normalize(vmin=vmin, vmax=vmax, clip=True)

        plt.figure(figsize=(10, 10))
        plt.imshow(data, cmap=cmap_name, norm=norm)
        plt.axis('off')
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)
        plt.close()
        print(f"✅ Convertido: {filename} [Paleta: {cmap_name}]")


def rgb_to_png(rgb_array, save_path):
    """
    Guarda un array RGB (3, H, W) como PNG en color natural.
    rgb_array: array numpy de forma (3, H, W) con valores normalizados [0, 1].
    """
    rgb = np.transpose(rgb_array, (1, 2, 0))  # (H, W, 3)
    rgb = np.clip(rgb, 0, 1)

    plt.figure(figsize=(12, 12))
    plt.imshow(rgb)
    plt.axis('off')
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)
    plt.close()
    print(f"✅ PNG guardado: {save_path}")


print("✅ Funciones auxiliares cargadas.")

---
## ⚙️ CONFIGURACIÓN
**Edita las rutas abajo** según la ubicación de tu shapefile en Google Drive.

In [ ]:
# =============================================================================
# CELDA 5: CONFIGURACIÓN — RUTAS Y PARÁMETROS
# =============================================================================

# Ruta del shapefile (ya configurada)
SHAPEFILE_PATH = "/content/drive/MyDrive/Tesis/GEE Murcott/ROI/Roigeneral.zip"

# Directorio base para exportación (se crea automáticamente al lado del shapefile)
BASE_DIR = os.path.join(os.path.dirname(SHAPEFILE_PATH), "Fase1")

# ⚙️ CONFIGURACIÓN DE BÚSQUEDA
# Cambia estos valores según lo que quieras consultar
MES = 4               # Mes (1=Enero, 2=Febrero, ..., 4=Abril, ... 12=Diciembre)
ANIO_INICIO = 2025    # Año inicial
ANIO_FIN = 2025       # Año final (puede ser el mismo si solo quieres un año)
# Filtro de nubes (opcional): descomenta la línea si quieres filtrar
# CLOUD_FILTER = {"eo:cloud_cover": {"lt": 10}}  # Solo <10% nubes
CLOUD_FILTER = None  # None = sin filtro (trae todo)

# ─── No tocar desde aquí ───────────────────────────────────────────────────
import calendar
FECHAS = []
MES_NOMBRE = ["", "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
               "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
for anio in range(ANIO_INICIO, ANIO_FIN + 1):
    ultimo_dia = calendar.monthrange(anio, MES)[1]
    desde = f"{anio}-{MES:02d}-01"
    hasta = f"{anio}-{MES:02d}-{ultimo_dia}"
    FECHAS.append((desde, hasta, f"{MES_NOMBRE[MES]} {anio}"))

print("✅ Configuración cargada.")
print(f"   Shapefile: {SHAPEFILE_PATH}")
print(f"   Período: {FECHAS[0][2] if FECHAS else 'N/A'}")
if CLOUD_FILTER:
    print(f"   Filtro nubes: < {CLOUD_FILTER['eo:cloud_cover']['lt']}%")
else:
    print(f"   Filtro nubes: NINGUNO (todas las imágenes)")
print(f"   Exportar a: {BASE_DIR}")

In [ ]:
# =============================================================================
# CELDA 6: CARGAR SHAPEFILE Y CALCULAR BOUNDING BOX
# =============================================================================

print("🔍 Cargando shapefile...")
gdf = gpd.read_file(SHAPEFILE_PATH)

print(f"✅ Shapefile cargado: {len(gdf)} feature(s)")
print(f"   Columnas: {list(gdf.columns)}")
print(f"   CRS original: {gdf.crs}")

# Asegurar CRS en WGS84 (lon/lat) para STAC
if gdf.crs and gdf.crs.is_geographic:
    gdf_geo = gdf
else:
    gdf_geo = gdf.to_crs("EPSG:4326")
    print(f"   CRS reproyectado a: EPSG:4326")

# Calcular bounding box [minx, miny, maxx, maxy]
bbox = gdf_geo.total_bounds
print(f"\n📐 Bounding Box (lon_min, lat_min, lon_max, lat_max):")
print(f"   {bbox}")
print(f"   ({bbox[0]:.4f}, {bbox[1]:.4f}, {bbox[2]:.4f}, {bbox[3]:.4f})")

# Visualizar el shapefile
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
gdf_geo.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=1.5)
ax.set_title("ROI — Área de Estudio Mandarina Murcott", fontsize=12)
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
plt.tight_layout()
plt.show()

# Crear directorios de salida
os.makedirs(f"{BASE_DIR}/PNG", exist_ok=True)
os.makedirs(f"{BASE_DIR}/GeoTIFF", exist_ok=True)
print(f"✅ Directorios creados:")
print(f"   PNG → {BASE_DIR}/PNG/")
print(f"   GeoTIFF → {BASE_DIR}/GeoTIFF/")

---
## 🔎 BÚSQUEDA DE IMÁGENES SENTINEL-2

In [ ]:
# =============================================================================
# CELDA 7: BÚSQUEDA EN CATÁLOGO STAC + CONTEO + CLASIFICACIÓN DE NUBES
# =============================================================================

print("🔍 Conectando al catálogo Earth Search (AWS)...")
catalog = Client.open("https://earth-search.aws.element84.com/v1")
print("✅ Conexión exitosa.")

resultados_totales = []

for fecha_inicio, fecha_fin, label in FECHAS:
    print(f"\n📅 {label} ({fecha_inicio} → {fecha_fin}):")
    params = dict(
        collections=["sentinel-2-l2a"],
        bbox=list(bbox),
        datetime=f"{fecha_inicio}/{fecha_fin}",
    )
    if CLOUD_FILTER:
        params["query"] = CLOUD_FILTER
    search = catalog.search(**params)
    items = list(search.items())
    print(f"   Imágenes encontradas: {len(items)}")

    for item in items:
        cloud = item.properties.get("eo:cloud_cover", None)
        if cloud is None:
            estado = "Sin dato"
            cloud_val = -1
        elif cloud < 20:
            estado = "Despejada"
            cloud_val = cloud
        elif cloud < 60:
            estado = "Parcial"
            cloud_val = cloud
        else:
            estado = "Nublada"
            cloud_val = cloud

        resultados_totales.append({
            "fecha": item.datetime.strftime("%Y-%m-%d"),
            "id": item.id,
            "nubes_%": cloud_val,
            "estado": estado,
            "mes": label,
        })

df_resultados = pd.DataFrame(resultados_totales)

if len(df_resultados) == 0:
    print("\n❌ No se encontraron imágenes. Verifica el bbox y las fechas.")
else:
    print("\n" + "=" * 70)
    print("📊 RESUMEN POR FECHA:")
    print("=" * 70)

    for (mes, fecha), group in df_resultados.groupby(["mes", "fecha"]):
        nubes_prom = group["nubes_%"].mean()
        num_img = len(group)

        if nubes_prom < 0:
            icon = "❓"
            estado_txt = "Sin dato de nubes"
        elif nubes_prom < 20:
            icon = "🌤️"
            estado_txt = f"Despejada ({nubes_prom:.0f}%)"
        elif nubes_prom < 60:
            icon = "⛅"
            estado_txt = f"Parcial ({nubes_prom:.0f}%)"
        else:
            icon = "☁️"
            estado_txt = f"Nublada ({nubes_prom:.0f}%)"

        print(f"  {fecha} ({mes}): {num_img} img | {icon} {estado_txt}")

    print(f"\n{'=' * 70}")
    print(f"📈 TOTAL: {len(df_resultados)} escenas en el período.")
    print(f"{'=' * 70}")

---
## 📦 CARGA Y VISUALIZACIÓN DE IMÁGENES

In [ ]:
# =============================================================================
# CELDA 8: CARGA DEL DATA CUBE (SOLO BANDAS RGB)
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay imágenes para cargar. Ejecuta la celda anterior primero.")
else:
    print("📦 Cargando Data Cube desde STAC...")
    print(f"   Bandas: blue (B2), green (B3), red (B4)")
    print(f"   Resolución: ~10m (0.0001°)")

    # Re-coleccionar items para el load (necesitamos la lista completa)
    items_list = []
    for fecha_inicio, fecha_fin, label in FECHAS:
        params = dict(
            collections=["sentinel-2-l2a"],
            bbox=list(bbox),
            datetime=f"{fecha_inicio}/{fecha_fin}",
        )
        if CLOUD_FILTER:
            params["query"] = CLOUD_FILTER
        search = catalog.search(**params)
        items_list.extend(list(search.items()))

    print(f"   Total de escenas a cargar: {len(items_list)}")

    ds = load(
        items_list,
        bands=["blue", "green", "red"],
        bbox=list(bbox),
        crs="EPSG:4326",
        resolution=0.0001,
        groupby="solar_day",
        chunks={'time': 1, 'x': 512, 'y': 512},
    )

    print(f"✅ Data Cube cargado exitosamente.")
    print(f"   Dimensiones: {dict(ds.sizes)}")
    print(f"   Fechas disponibles: {ds.sizes['time']}")

In [ ]:
# =============================================================================
# CELDA 9: VISUALIZACIÓN RGB — COLOR NATURAL
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay datos para visualizar.")
else:
    print("🖼️ Generando visualización RGB en color natural...")

    # Normalizar bandas (Sentinel-2 L2A viene escalado por 10000)
    red_band = ds["red"].values / 10000.0
    green_band = ds["green"].values / 10000.0
    blue_band = ds["blue"].values / 10000.0

    n_times = ds.dims["time"]
    n_cols = min(3, n_times)
    n_rows = max(1, (n_times + n_cols - 1) // n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    if n_times == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for t in range(n_times):
        ax = axes[t]

        # Componer RGB (red, green, blue → canales R, G, B)
        rgb = np.stack([red_band[t], green_band[t], blue_band[t]], axis=-1)
        rgb = np.clip(rgb, 0, 1)

        ax.imshow(rgb)
        fecha = str(ds.time.values[t])[:10]

        # Buscar clasificación de nubes para esta fecha
        estado = "N/D"
        if fecha in df_resultados["fecha"].values:
            subset = df_resultados[df_resultados["fecha"] == fecha]
            nubes = subset["nubes_%"].mean()
            if nubes < 0:
                estado = "❓ Sin dato"
            elif nubes < 20:
                estado = f"🌤️ {nubes:.0f}%"
            elif nubes < 60:
                estado = f"⛅ {nubes:.0f}%"
            else:
                estado = f"☁️ {nubes:.0f}%"

        ax.set_title(f"{fecha} | {estado}", fontsize=10, fontweight='bold')
        ax.axis('off')

    # Ocultar ejes no usados
    for t in range(n_times, len(axes)):
        axes[t].axis('off')

    plt.suptitle(
        "Sentinel-2 L2A — Color Natural (RGB) — Mandarina Murcott",
        fontsize=14, y=1.02
    )
    plt.tight_layout()

    # Guardar grid completo
    grid_path = f"{BASE_DIR}/PNG/Grid_RGB_Todas_las_fechas.png"
    plt.savefig(grid_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✅ Grid guardado en: {grid_path}")

---
## 💾 EXPORTACIÓN DE IMÁGENES

In [ ]:
# =============================================================================
# CELDA 10: EXPORTAR A GeoTIFF
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay datos para exportar.")
else:
    print("💾 Exportando GeoTIFFs (3 bandas: R, G, B)...")

    for t in range(ds.dims["time"]):
        fecha = str(ds.time.values[t])[:10]
        geotiff_path = f"{BASE_DIR}/GeoTIFF/Mandarina_{fecha}_RGB.tif"

        # Stackear las 3 bandas y exportar con metadatos espaciales
        rgb_da = (
            ds.isel(time=t)[["red", "green", "blue"]]
            .to_array(dim="band")
        )
        rgb_da.rio.to_raster(geotiff_path)
        print(f"   ✅ GeoTIFF: Mandarina_{fecha}_RGB.tif")

    print(f"\n📁 Todos los GeoTIFFs guardados en: {BASE_DIR}/GeoTIFF/")

In [ ]:
# =============================================================================
# CELDA 11: EXPORTAR A PNG (IMÁGENES INDIVIDUALES)
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay datos para exportar.")
else:
    print("🖼️ Exportando PNGs individuales...")

    for t in range(ds.dims["time"]):
        fecha = str(ds.time.values[t])[:10]
        png_path = f"{BASE_DIR}/PNG/Mandarina_{fecha}_RGB.png"

        # Componer y guardar RGB
        rgb = np.stack([
            ds["red"].values[t] / 10000.0,
            ds["green"].values[t] / 10000.0,
            ds["blue"].values[t] / 10000.0,
        ], axis=-1)
        rgb = np.clip(rgb, 0, 1)

        plt.figure(figsize=(10, 10))
        plt.imshow(rgb)
        plt.title(f"Mandarina Murcott — {fecha}", fontsize=12)
        plt.axis('off')
        plt.savefig(png_path, bbox_inches='tight', pad_inches=0, dpi=300)
        plt.close()
        print(f"   ✅ PNG: Mandarina_{fecha}_RGB.png")

    print(f"\n📁 Todos los PNGs guardados en: {BASE_DIR}/PNG/")

In [ ]:
# =============================================================================
# CELDA 12: GENERAR REPORTE TXT
# =============================================================================

ruta_reporte = f"{BASE_DIR}/Reporte_Fase1_Abril.txt"

with open(ruta_reporte, "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("   FASE 1 — REPORTE DE IMÁGENES SENTINEL-2\n")
    f.write("   Mandarina Murcott — Abril 2025 y 2026\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"  Bounding Box: ")
    f.write(f"[{bbox[0]:.4f}, {bbox[1]:.4f}, {bbox[2]:.4f}, {bbox[3]:.4f}]\n\n")

    for label in df_resultados["mes"].unique():
        f.write(f"📅 {label}:\n")
        subset = df_resultados[df_resultados["mes"] == label]
        for (fecha,), grupo in subset.groupby("fecha"):
            nubes_prom = grupo["nubes_%"].mean()
            if nubes_prom < 0:
                icon = "Sin dato"
            elif nubes_prom < 20:
                icon = "Despejada"
            elif nubes_prom < 60:
                icon = "Parcial"
            else:
                icon = "Nublada"
            f.write(f"    - {fecha}: {len(grupo)} img [{icon}]\n")
        f.write(f"  Subtotal: {len(subset)} imágenes\n\n")

    f.write(f"  TOTAL GENERAL: {len(df_resultados)} imágenes\n")
    f.write("\n" + "=" * 60 + "\n")
    f.write("ARCHIVOS EXPORTADOS:\n")
    png_count = len(glob.glob(f"{BASE_DIR}/PNG/[!Grid]*.png"))
    tif_count = len(glob.glob(f"{BASE_DIR}/GeoTIFF/*.tif"))
    f.write(f"  PNG: {png_count} archivos\n")
    f.write(f"  GeoTIFF: {tif_count} archivos\n")
    f.write(f"  Ruta: {BASE_DIR}\n")
    f.write("=" * 60 + "\n")

print(f"✅ Reporte guardado en: {ruta_reporte}")
print("=" * 60)
print("   🎉 FASE 1 COMPLETADA")
print("=" * 60)